# RAG 검색 성능 평가

동일한 평가셋에서 keyword only, vector only, hybrid 검색을 비교합니다.

평가 지표는 `Hit Rate@3`, `Recall@5`, `MRR@5`, `nDCG@5`입니다. 이 노트북은 검색 성능만 평가하며 LLM 답변 품질은 포함하지 않습니다.

In [1]:
import json
import math
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

WORKING_DIR = Path.cwd().resolve()
BACKEND_ROOT = WORKING_DIR if (WORKING_DIR / 'data').is_dir() and (WORKING_DIR / 'notebooks').is_dir() else WORKING_DIR.parent
if not ((BACKEND_ROOT / 'data').is_dir() and (BACKEND_ROOT / 'notebooks').is_dir()):
    raise RuntimeError('프로젝트 루트 또는 notebooks 폴더에서 노트북을 실행해야 합니다.')

if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

load_dotenv(BACKEND_ROOT / '.env')
DATASET_PATH = BACKEND_ROOT / 'notebooks' / 'data' / '01_retrieval_eval_set.json'
TOP_K = 5
HIT_K = 3

print(f'backend root: {BACKEND_ROOT}')
print(f'evaluation set: {DATASET_PATH.name}')

backend root: /home/sms/openclaw_file/RAIchU/SKN25-4th-6Team/backend
evaluation set: 01_retrieval_eval_set.json


In [2]:
from src.retrieval import (
    infer_filters_from_question,
    retrieve_cards_hybrid,
    retrieve_cards_keyword,
    retrieve_cards_vector,
)
from src.service import load_app_state

with DATASET_PATH.open(encoding='utf-8') as file:
    dataset = json.load(file)

app_state = load_app_state(
    data_dir=BACKEND_ROOT / 'data' / 'cards',
    category_config_path=BACKEND_ROOT / 'rag_config' / 'card_category_rules.json',
    rag_config_path=BACKEND_ROOT / 'rag_config' / 'rag_settings.json',
    synonyms_config_path=BACKEND_ROOT / 'rag_config' / 'synonyms.json',
    rag_artifacts_dir=BACKEND_ROOT / 'vector_store',
    mbti_config_path=BACKEND_ROOT / 'rag_config' / 'mbti_rules.json',
)

vector_ready = bool(os.getenv('OPENAI_API_KEY')) and app_state.vector_store is not None
print(f"cards: {len(app_state.cards)}, evaluation cases: {len(dataset['cases'])}")
print(f'vector search available: {vector_ready}')
if not vector_ready:
    print('WARNING: OPENAI_API_KEY가 없거나 벡터 인덱스를 읽지 못했습니다. vector는 빈 결과이고 hybrid는 keyword-only로 동작합니다.')

cards: 106, evaluation cases: 30
vector search available: True


In [3]:
def card_ids(results):
    return [card.get('_file', '').removesuffix('.json') for _, card in results]

def hit_rate_at_k(result_ids, relevant_ids, k):
    return int(bool(set(result_ids[:k]) & set(relevant_ids)))

def recall_at_k(result_ids, relevant_ids, k):
    relevant = set(relevant_ids)
    return len(set(result_ids[:k]) & relevant) / len(relevant) if relevant else 0.0

def reciprocal_rank_at_k(result_ids, relevant_ids, k):
    relevant = set(relevant_ids)
    for rank, card_id in enumerate(result_ids[:k], start=1):
        if card_id in relevant:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(result_ids, relevance_grades, k):
    dcg = sum(
        (2 ** relevance_grades.get(card_id, 0) - 1) / math.log2(rank + 1)
        for rank, card_id in enumerate(result_ids[:k], start=1)
    )
    ideal_grades = sorted(relevance_grades.values(), reverse=True)[:k]
    idcg = sum(
        (2 ** grade - 1) / math.log2(rank + 1)
        for rank, grade in enumerate(ideal_grades, start=1)
    )
    return dcg / idcg if idcg else 0.0

def inferred_filters(query):
    return infer_filters_from_question(
        query,
        app_state.banks,
        app_state.all_categories,
        app_state.fee_bands,
        synonyms=app_state.synonyms,
    )

def retrieve(query, method):
    filters = inferred_filters(query)
    common = {
        'cards': app_state.cards,
        'query': query,
        'top_k': TOP_K,
        'banks': filters['banks'],
        'categories': filters['categories'],
        'fee_bands': filters['fee_bands'],
    }
    if method == 'keyword':
        return retrieve_cards_keyword(**common, synonyms=app_state.synonyms)
    if method == 'vector':
        return retrieve_cards_vector(
            **common,
            vector_store=app_state.vector_store,
            embedding_model=app_state.rag_settings['embedding_model'],
            similarity_threshold=app_state.rag_settings['similarity_threshold'],
        ) if app_state.vector_store else []
    if method == 'hybrid':
        return retrieve_cards_hybrid(
            **common,
            vector_store=app_state.vector_store,
            embedding_model=app_state.rag_settings['embedding_model'],
            similarity_threshold=app_state.rag_settings['similarity_threshold'],
            synonyms=app_state.synonyms,
        )
    raise ValueError(f'unsupported method: {method}')

In [4]:
METHODS = ('vector', 'keyword', 'hybrid')
case_results = []
summary = {method: {'hit_rate_at_3': [], 'recall_at_5': [], 'mrr_at_5': [], 'ndcg_at_5': []} for method in METHODS}

for case in dataset['cases']:
    for method in METHODS:
        try:
            result_ids = card_ids(retrieve(case['query'], method))
            metrics = {
                'hit_rate_at_3': hit_rate_at_k(result_ids, case['relevant_card_ids'], HIT_K),
                'recall_at_5': recall_at_k(result_ids, case['relevant_card_ids'], TOP_K),
                'mrr_at_5': reciprocal_rank_at_k(result_ids, case['relevant_card_ids'], TOP_K),
                'ndcg_at_5': ndcg_at_k(result_ids, case['relevance_grades'], TOP_K),
            }
            for metric, value in metrics.items():
                summary[method][metric].append(value)
            case_results.append({
                'case_id': case['id'],
                'query_type': case['query_type'],
                'method': method,
                'result_ids': result_ids,
                **metrics,
            })
        except Exception as error:
            case_results.append({
                'case_id': case['id'],
                'query_type': case['query_type'],
                'method': method,
                'error': str(error),
            })

comparison = {
    method: {metric: round(sum(values) / len(values), 4) if values else None for metric, values in metrics.items()}
    for method, metrics in summary.items()
}

print('전체 비교 결과')
print(f"{'method':<10} {'Hit@3':>8} {'Recall@5':>10} {'MRR@5':>8} {'nDCG@5':>9}")
for method in METHODS:
    metrics = comparison[method]
    print(f"{method:<10} {metrics['hit_rate_at_3']:>8.4f} {metrics['recall_at_5']:>10.4f} {metrics['mrr_at_5']:>8.4f} {metrics['ndcg_at_5']:>9.4f}")
errors = [row for row in case_results if 'error' in row]
print(f'completed rows: {len(case_results)}, errors: {len(errors)}')
if errors:
    print(json.dumps(errors[:5], ensure_ascii=False, indent=2))

전체 비교 결과
method        Hit@3   Recall@5    MRR@5    nDCG@5
vector       0.6667     0.5833   0.5472    0.5146
keyword      0.7667     0.7672   0.6772    0.6480
hybrid       0.7667     0.7600   0.6383    0.6394
completed rows: 90, errors: 0


In [5]:
print('\n질문별 실패 사례:')
for row in case_results:
    if 'error' not in row and row['hit_rate_at_3'] == 0:
        case = next(item for item in dataset['cases'] if item['id'] == row['case_id'])
        print(f"- [{row['method']}] {row['case_id']}: {case['query']}")
        print(f"  result: {row['result_ids']}")
        print(f"  expected: {case['relevant_card_ids']}")

if not vector_ready:
    print('\n해석 주의: vector 검색을 사용할 API 키가 없어서 vector/hybrid 수치를 공정하게 비교할 수 없습니다.')
else:
    best_method = max(comparison, key=lambda method: comparison[method]['ndcg_at_5'])
    print(f"\nnDCG@5 기준 최고 방식: {best_method}")


질문별 실패 사례:
- [vector] ret_002: 전월 실적 조건 없이 캐시백을 주는 IBK 카드를 알려줘
  result: ['IBK_Everyday_Joy_Credit', 'IBK_i-Mileage', 'IBK_DailyWith', 'IBK_KPass(Credit)', 'IBK_Bliss5']
  expected: ['IBK_IBK-Hybrid']
- [keyword] ret_002: 전월 실적 조건 없이 캐시백을 주는 IBK 카드를 알려줘
  result: ['IBK_iAll', 'IBK_i-Mileage', 'IBK_DailyWith', 'IBK_Bliss5', 'IBK_Everyday_Joy_Credit']
  expected: ['IBK_IBK-Hybrid']
- [hybrid] ret_002: 전월 실적 조건 없이 캐시백을 주는 IBK 카드를 알려줘
  result: ['IBK_i-Mileage', 'IBK_iAll', 'IBK_DailyWith', 'IBK_Everyday_Joy_Credit', 'IBK_Bliss5']
  expected: ['IBK_IBK-Hybrid']
- [vector] ret_003: 배달과 카페 할인 혜택이 큰 롯데카드를 추천해줘
  result: ['Lotte_Digiloca_MONACO', 'Lotte_Digiloca_LAS_VEGAS', 'Lotte_LOCA_LIKIT_Shop', 'Lotte_LOCA_LIKIT_Eat', 'Lotte_Digiloca_PARIS']
  expected: ['Lotte_LOCA_LIKIT_Eat']
- [vector] ret_006: K-패스 교통 할인 카드를 추천해줘
  result: ['NH_Olbareun_OIL&PASS', 'Kookmin_AlphaOne_20210923', 'Samsung_iD_ON', 'BC_K_FRIST', 'NH_Olbareun_FLEX']
  expected: ['IBK_KPass(Credit)', 'Kookmin_K-Pass_20240424'

## Hybrid 가중치 비교

동일한 평가셋에서 vector:keyword 가중치를 5:5, 6:4, 7:3으로 바꿔 비교합니다. 검색 구현이나 기본 설정은 변경하지 않습니다.

In [6]:
HYBRID_WEIGHTS = {
    'hybrid_5_5': (0.5, 0.5),
    'hybrid_6_4': (0.6, 0.4),
    'hybrid_7_3': (0.7, 0.3),
}

def retrieve_hybrid_weighted(query, vector_weight, keyword_weight):
    filters = inferred_filters(query)
    return retrieve_cards_hybrid(
        cards=app_state.cards,
        query=query,
        top_k=TOP_K,
        banks=filters['banks'],
        categories=filters['categories'],
        fee_bands=filters['fee_bands'],
        vector_store=app_state.vector_store,
        embedding_model=app_state.rag_settings['embedding_model'],
        similarity_threshold=app_state.rag_settings['similarity_threshold'],
        synonyms=app_state.synonyms,
        vector_weight=vector_weight,
        keyword_weight=keyword_weight,
    )

weight_summary = {
    name: {'hit_rate_at_3': [], 'recall_at_5': [], 'mrr_at_5': [], 'ndcg_at_5': []}
    for name in HYBRID_WEIGHTS
}
weight_errors = []

for case in dataset['cases']:
    for name, (vector_weight, keyword_weight) in HYBRID_WEIGHTS.items():
        try:
            result_ids = card_ids(retrieve_hybrid_weighted(case['query'], vector_weight, keyword_weight))
            metrics = {
                'hit_rate_at_3': hit_rate_at_k(result_ids, case['relevant_card_ids'], HIT_K),
                'recall_at_5': recall_at_k(result_ids, case['relevant_card_ids'], TOP_K),
                'mrr_at_5': reciprocal_rank_at_k(result_ids, case['relevant_card_ids'], TOP_K),
                'ndcg_at_5': ndcg_at_k(result_ids, case['relevance_grades'], TOP_K),
            }
            for metric, value in metrics.items():
                weight_summary[name][metric].append(value)
        except Exception as error:
            weight_errors.append({'case_id': case['id'], 'weight': name, 'error': str(error)})

weight_comparison = {
    name: {metric: round(sum(values) / len(values), 4) if values else None for metric, values in metrics.items()}
    for name, metrics in weight_summary.items()
}

print('Hybrid 가중치 비교 결과')
print(f"{'vector:keyword':<18} {'Hit@3':>8} {'Recall@5':>10} {'MRR@5':>8} {'nDCG@5':>9}")
for name in HYBRID_WEIGHTS:
    metrics = weight_comparison[name]
    vector_weight, keyword_weight = HYBRID_WEIGHTS[name]
    label = f'{vector_weight:.1f}:{keyword_weight:.1f}'
    print(f"{label:<18} {metrics['hit_rate_at_3']:>8.4f} {metrics['recall_at_5']:>10.4f} {metrics['mrr_at_5']:>8.4f} {metrics['ndcg_at_5']:>9.4f}")
print(f'completed rows: {len(dataset["cases"]) * len(HYBRID_WEIGHTS)}, errors: {len(weight_errors)}')
if weight_errors:
    print(json.dumps(weight_errors[:5], ensure_ascii=False, indent=2))

Hybrid 가중치 비교 결과
vector:keyword        Hit@3   Recall@5    MRR@5    nDCG@5
0.5:0.5              0.7667     0.7867   0.6217    0.6324
0.6:0.4              0.7667     0.7600   0.6383    0.6394
0.7:0.3              0.7000     0.7156   0.6244    0.6147
completed rows: 90, errors: 0


## Keyword 우세 Hybrid 가중치 비교

앞선 5:5, 6:4, 7:3 비교와 동일한 조건에서 keyword 비중을 높인 4:6, 3:7을 추가로 평가합니다.

In [7]:
KEYWORD_HEAVY_WEIGHTS = {
    'hybrid_4_6': (0.4, 0.6),
    'hybrid_3_7': (0.3, 0.7),
}

keyword_heavy_summary = {
    name: {'hit_rate_at_3': [], 'recall_at_5': [], 'mrr_at_5': [], 'ndcg_at_5': []}
    for name in KEYWORD_HEAVY_WEIGHTS
}
keyword_heavy_errors = []

for case in dataset['cases']:
    for name, (vector_weight, keyword_weight) in KEYWORD_HEAVY_WEIGHTS.items():
        try:
            result_ids = card_ids(retrieve_hybrid_weighted(case['query'], vector_weight, keyword_weight))
            metrics = {
                'hit_rate_at_3': hit_rate_at_k(result_ids, case['relevant_card_ids'], HIT_K),
                'recall_at_5': recall_at_k(result_ids, case['relevant_card_ids'], TOP_K),
                'mrr_at_5': reciprocal_rank_at_k(result_ids, case['relevant_card_ids'], TOP_K),
                'ndcg_at_5': ndcg_at_k(result_ids, case['relevance_grades'], TOP_K),
            }
            for metric, value in metrics.items():
                keyword_heavy_summary[name][metric].append(value)
        except Exception as error:
            keyword_heavy_errors.append({'case_id': case['id'], 'weight': name, 'error': str(error)})

keyword_heavy_comparison = {
    name: {metric: round(sum(values) / len(values), 4) if values else None for metric, values in metrics.items()}
    for name, metrics in keyword_heavy_summary.items()
}

print('Keyword 우세 Hybrid 가중치 비교 결과')
print(f"{'vector:keyword':<18} {'Hit@3':>8} {'Recall@5':>10} {'MRR@5':>8} {'nDCG@5':>9}")
for name in KEYWORD_HEAVY_WEIGHTS:
    metrics = keyword_heavy_comparison[name]
    vector_weight, keyword_weight = KEYWORD_HEAVY_WEIGHTS[name]
    label = f'{vector_weight:.1f}:{keyword_weight:.1f}'
    print(f"{label:<18} {metrics['hit_rate_at_3']:>8.4f} {metrics['recall_at_5']:>10.4f} {metrics['mrr_at_5']:>8.4f} {metrics['ndcg_at_5']:>9.4f}")
print(f'completed rows: {len(dataset["cases"]) * len(KEYWORD_HEAVY_WEIGHTS)}, errors: {len(keyword_heavy_errors)}')
if keyword_heavy_errors:
    print(json.dumps(keyword_heavy_errors[:5], ensure_ascii=False, indent=2))

Keyword 우세 Hybrid 가중치 비교 결과
vector:keyword        Hit@3   Recall@5    MRR@5    nDCG@5
0.4:0.6              0.7667     0.8117   0.6483    0.6478
0.3:0.7              0.7667     0.8117   0.6483    0.6413
completed rows: 60, errors: 0
